# TrendScope Advanced ML: S-BERT + UMAP + HDBSCAN (Fixed Visualization)
**Objective:** Upgrade trend clustering from simple Keyword Matching (TF-IDF) to Semantic Understanding (Sentence Transformers).

**Steps:**
1.  Setup & Install Libraries (GPU Optimized)
2.  Load Data form Google Drive
3.  Generate Embeddings (S-BERT)
4.  Dimensionality Reduction (UMAP fitted for inference)
5.  Density Clustering (HDBSCAN prediction-enabled)
6.  Evaluate **(Cluster Metrics & Distribution)**
7.  Visualize **(Fixed 2D Plot)**
8.  Save Model (Production Ready)

In [ ]:
# 1. Install Libraries
!pip install -U sentence-transformers umap-learn hdbscan matplotlib pandas

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Path where you uploaded your CSVs
# IMPORTANT: Create a folder named 'TrendScope_Data' in your Drive root
DATA_PATH = '/content/drive/My Drive/TrendScope_Data'

# Load Historical Data
try:
    df_2020 = pd.read_csv(os.path.join(DATA_PATH, 'filtered_2020_english.csv'))
    print(f"Loaded 2020 Data: {len(df_2020)} rows")
except Exception as e:
    print(f"Warning: Could not load 2020 data. {e}")
    df_2020 = pd.DataFrame()

# Load Live Data (Optional)
try:
    df_live = pd.read_csv(os.path.join(DATA_PATH, 'live_data_export.csv'))
    print(f"Loaded Live Data: {len(df_live)} rows")
except Exception as e:
    print(f"Warning: Could not load live data. Using only historical. {e}")
    df_live = pd.DataFrame()

# Merge
df = pd.concat([df_2020, df_live], ignore_index=True)
# Limit for Performance (50k is sufficient to learn topics)
df = df.head(50000)
print(f"Total Training Data: {len(df)} rows")

# Text Preprocessing (Simple)
df['text'] = df['title'].fillna('') + " " + df['tags'].fillna('')
data = df['text'].tolist()
print(f"Sample: {data[:2]}")

In [ ]:
# 3. Generate Embeddings (Sentence-BERT)
from sentence_transformers import SentenceTransformer

# Load Semantic Model (MiniLM is fast & good)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding data (this may take a minute on GPU)...")
embeddings = embedding_model.encode(data, show_progress_bar=True)
print(f"Embeddings Shape: {embeddings.shape}")

In [ ]:
# 4. Reduce Dimensions (UMAP - Production Ready)
import umap

# Key Change: 'init=random' ensures we don't get stuck on Spectral Initialization Failure
print("Running UMAP...")
umap_model = umap.UMAP(n_neighbors=15, 
                       n_components=5, 
                       metric='cosine', 
                       init='random',
                       random_state=42) 

umap_embeddings = umap_model.fit_transform(embeddings)
print("UMAP Complete.")

In [ ]:
# 5. Cluster (HDBSCAN - Production Ready)
import hdbscan
from sklearn.metrics import silhouette_score

print("Running HDBSCAN...")
# Key Change: prediction_data=True allows us to predict on NEW videos later!
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=15,
                                metric='euclidean', 
                                cluster_selection_method='eom',
                                prediction_data=True)

labels = hdbscan_model.fit_predict(umap_embeddings)

# Calculate Sensitivity (Silhouette Score)
score = silhouette_score(umap_embeddings, labels)
print(f"\n>>> NEW SILHOUETTE SCORE (5D): {score:.4f} <<<")

# Check Cluster Stats
import numpy as np
unique, counts = np.unique(labels, return_counts=True)
n_clusters = len(unique) - (1 if -1 in unique else 0)
n_noise = counts[unique == -1][0] if -1 in unique else 0

print(f"Clusters Found: {n_clusters}")
print(f"Noise Points (Unclassified): {n_noise} / {len(labels)} ({n_noise/len(labels)*100:.1f}%)")

# Calculate Avg/Max Cluster Size
cluster_sizes = counts[unique != -1]
print(f"Average Cluster Size: {np.mean(cluster_sizes):.1f} videos")
print(f"Largest Cluster: {np.max(cluster_sizes)} videos")
print(f"Smallest Cluster: {np.min(cluster_sizes)} videos")

In [ ]:
# 6. Detailed Metrics Visualization (Answer the Critic!)
import matplotlib.pyplot as plt

# Plot Histogram of Cluster Sizes
plt.figure(figsize=(10, 5))
plt.hist(cluster_sizes, bins=50, log=True, color='skyblue', edgecolor='black')
plt.title('Cluster Size Distribution (Log Scale)')
plt.xlabel('Number of Videos in Cluster')
plt.ylabel('Frequency (Number of Clusters)')
plt.grid(axis='y', alpha=0.5)
plt.savefig(os.path.join(DATA_PATH, 'cluster_distribution_plot.png'))
print("Cluster Distribution Plot saved!")
plt.show()

In [ ]:
# 7. Fixed 2D Visualization (The Pretty Picture)
# We re-run UMAP specifically for 2D visualization with random init to avoid 'blob' issue
print("Generating 2D Projection for Plotting...")
if umap_embeddings.shape[1] > 2:
    # Use init='random' specifically here
    vis_embeddings = umap.UMAP(n_neighbors=15, 
                               n_components=2, 
                               min_dist=0.1, 
                               metric='cosine',
                               init='random',
                               random_state=42).fit_transform(embeddings)
else:
    vis_embeddings = umap_embeddings

plt.figure(figsize=(12, 8))
scatter = plt.scatter(vis_embeddings[:, 0], 
                      vis_embeddings[:, 1], 
                      c=labels, 
                      cmap='Spectral', 
                      s=10, 
                      alpha=0.6)
plt.colorbar(scatter, label='Cluster ID')
plt.title(f'TrendScope Advanced Clusters (Score: {score:.2f})', fontsize=16)
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')

# Save plot to Drive
plt.savefig(os.path.join(DATA_PATH, 'advanced_cluster_plot_fixed.png'))
print("Fixed Plot saved to Google Drive!")
plt.show()

In [ ]:
# 8. Save Model for Backend
import joblib

print("Saving model objects...")

model_data = {
    "embedding_model_name": "all-MiniLM-L6-v2",
    "umap_model": umap_model,        # The fitted transformer
    "hdbscan_model": hdbscan_model,  # The fitted clusterer with prediction_data=True
    "cluster_meta": {} 
}

# Create simple metadata for clusters
df['cluster'] = labels
cluster_meta = {}
for cluster_id in unique:
    if cluster_id == -1: continue
    # Just counting size for now to keep file small
    cluster_meta[int(cluster_id)] = {"id": int(cluster_id), "count": int(counts[unique == cluster_id][0])}

model_data["cluster_meta"] = cluster_meta

joblib.dump(model_data, os.path.join(DATA_PATH, 'advanced_trend_model.pkl'))
print("SUCCESS: Production-Ready Model saved to Google Drive: advanced_trend_model.pkl")